In [1]:
"""
Flag top false-positive (FP) patterns from evidence fields.

Inputs (in the same folder as your metrics runner):
- Manual sheet (autofilled + reviewed), picks first existing among:
    manual_review_sheet_autofilled.csv
    Manual_Review_Sheet__Prefilled_RQ1.csv
    manual_review_sheet_prefilled_v2.csv
    manual_review_sheet_prefilled.csv
- cohort_table.csv (for stratum population sizes N_h)

Outputs (written to ...\Stratified Sample\Metrics):
- fp_patterns_yaml_weighted.csv
- fp_patterns_build_weighted.csv
- fp_patterns_at_weighted.csv

Each file lists: pattern, weighted_count, share, unweighted_n, example_repos (up to 5).
"""

import re
import math
import numpy as np
import pandas as pd
from pathlib import Path

# ---------- CONFIG ----------
BASE_DIR   = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample"
OUTPUT_DIR = r"C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics"

MANUAL_FILES = [
    "manual_review_sheet_autofilled.csv",
    "Manual_Review_Sheet__Prefilled_RQ1.csv",
    "manual_review_sheet_prefilled_v2.csv",
    "manual_review_sheet_prefilled.csv",
]
COHORT_FILE = "cohort_table.csv"

# ---------- HELPERS ----------
def first_existing(base, names):
    base = Path(base)
    for n in names:
        p = base / n
        if p.exists():
            return str(p)
    raise FileNotFoundError(f"None of these found in {base}: {names}")

def to_int01(x):
    if pd.isna(x): return None
    if isinstance(x, (int, np.integer)): return int(x != 0)
    if isinstance(x, float):
        if np.isnan(x): return None
        return int(x != 0.0)
    s = str(x).strip().lower()
    if s in {"1","true","t","yes","y"}: return 1
    if s in {"0","false","f","no","n"}: return 0
    return None

# stratified expansion weight per row for a given truth target
def compute_weights(df_in, truth_col, Nh_map):
    # only rows with labeled truth get weights
    labeled = df_in[df_in[truth_col].apply(lambda x: x in (0,1))].copy()
    nh = labeled.groupby("stratum").size().rename("n_h")
    nh_map = nh.to_dict()
    w = []
    for _, r in df_in.iterrows():
        if not (r.get(truth_col) in (0,1)):
            w.append(np.nan); continue
        N_h = Nh_map.get(r["stratum"])
        n_h = nh_map.get(r["stratum"])
        if not N_h or not n_h:
            w.append(np.nan)
        else:
            w.append(N_h / n_h)
    return np.array(w, dtype="float64")

# ---------- LOAD ----------
manual_path = first_existing(BASE_DIR, MANUAL_FILES)
cohort_path = str(Path(BASE_DIR) / COHORT_FILE)

df = pd.read_csv(manual_path)
cohort = pd.read_csv(cohort_path)

for col in ["YAML_pred","Build_pred","AT_pred","yaml_signal_true","build_true","at_true","ci_runs_true"]:
    if col not in df.columns:
        df[col] = np.nan
    df[col] = df[col].apply(to_int01)

# population sizes by stratum
Nh_map = dict(zip(cohort["stratum"], cohort["N"]))
N_total = int(cohort["N"].sum())

# Ensure evidence fields exist
for c in ["yaml_paths","build_paths","androidtest_paths",
          "ci_command_evidence","device_setup_evidence","vendor_service_evidence",
          "full_name"]:
    if c not in df.columns:
        df[c] = ""

# ---------- PATTERN CATEGORIZERS ----------
# YAML FP: YAML_pred=1 & yaml_signal_true=0
RE_UNIT_ONLY = re.compile(r"(?:^|\s)(?:gradle[\w]*\s+)?(?:\S*unit\S*test|\S*test(?!(?:\S*connected)))", re.I)
RE_ANDROID_TOK = re.compile(r"android|emulator|avd|connectedandroidtest|connectedcheck|gcloud\s+firebase\s+test", re.I)
RE_VENDOR     = re.compile(r"(firebase\s+test\s+lab|browserstack|bstack|sauce|bitrise)", re.I)

def categorize_yaml_fp(row):
    cats = []
    ci_cmd = str(row.get("ci_command_evidence","") or "")
    dev    = str(row.get("device_setup_evidence","") or "")
    vend   = str(row.get("vendor_service_evidence","") or "")
    blob   = " ".join([ci_cmd, dev, vend]).lower()

    # Device setup present but no instrumentation run command found
    if dev.strip() and not ci_cmd.strip():
        cats.append("device_setup_only")

    # Vendor signals present but not clear run command
    if RE_VENDOR.search(vend) and ("gcloud firebase test android run" not in ci_cmd.lower()):
        cats.append("vendor_setup_only")

    # Unit-only test commands (no connected*)
    if RE_UNIT_ONLY.search(ci_cmd) and ("connected" not in ci_cmd.lower()):
        cats.append("unit_only_job")

    # Likely non-Android job: no android tokens but a "test" word present
    if ("test" in ci_cmd.lower()) and not RE_ANDROID_TOK.search(blob):
        cats.append("non_android_job")

    if not cats:
        cats.append("unknown_fp_yaml")
    return cats

# BUILD FP: Build_pred=1 & build_true=0
def categorize_build_fp(row):
    cats = []
    files = str(row.get("build_paths","") or "").lower()

    if ".gradle.kts" in files:
        cats.append("kotlin_dsl_mismatch")
    if "settings.gradle" in files:
        cats.append("multi_module_or_settings_only")
    if ".toml" in files:
        cats.append("versions_toml_only")
    if not cats:
        cats.append("generic_build_anchor_misfire")
    return cats

# AT FP: AT_pred=1 & at_true=0
def categorize_at_fp(row):
    cats = []
    files = str(row.get("androidtest_paths","") or "").lower()
    # no files captured at all -> likely detector overfired
    if not files.strip():
        cats.append("no_androidtest_files_captured")
    # only non-code files
    elif not any(ext in files for ext in [".kt",".java"]):
        cats.append("androidtest_non_code_only")
    else:
        cats.append("generic_at_fp")
    return cats

# ---------- CORE: COUNT & RANK ----------
def fp_table(df_in, fp_mask, truth_col, categorizer, Nh_map, top_k=30):
    d = df_in[fp_mask].copy()
    if d.empty:
        return pd.DataFrame(columns=["pattern","weighted_count","share","unweighted_n","example_repos"])

    # weights by the relevant truth col (so strata are consistent with that target)
    d["w"] = compute_weights(d, truth_col, Nh_map)

    # explode patterns
    d["patterns"] = d.apply(categorizer, axis=1)
    d = d.explode("patterns")
    d["patterns"] = d["patterns"].astype(str)

    # aggregate
    agg = (d.groupby("patterns")
             .agg(weighted_count=("w","sum"),
                  unweighted_n=("patterns","size"))
             .reset_index()
             .rename(columns={"patterns":"pattern"}))
    total_w = agg["weighted_count"].sum()
    agg["share"] = np.where(total_w>0, agg["weighted_count"]/total_w, np.nan)

    # examples (top 5 by weight per pattern)
    ex_rows = []
    for pat, grp in d.groupby("patterns"):
        ex = (grp.assign(w=grp["w"].fillna(0))
                .sort_values("w", ascending=False)
                .head(5))
        names = "; ".join([str(x) for x in ex["full_name"].fillna("").tolist() if str(x)])
        ex_rows.append({"pattern": pat, "example_repos": names})
    ex_df = pd.DataFrame(ex_rows)

    out = agg.merge(ex_df, on="pattern", how="left").sort_values("weighted_count", ascending=False)
    return out.head(top_k)

# ---------- BUILD TABLES ----------
# YAML FP patterns
mask_yaml_fp = (df["YAML_pred"]==1) & (df["yaml_signal_true"]==0)
yaml_tbl = fp_table(df, mask_yaml_fp, "yaml_signal_true", categorize_yaml_fp, Nh_map)

# Build FP patterns
mask_build_fp = (df["Build_pred"]==1) & (df["build_true"]==0)
build_tbl = fp_table(df, mask_build_fp, "build_true", categorize_build_fp, Nh_map)

# AT FP patterns
mask_at_fp = (df["AT_pred"]==1) & (df["at_true"]==0)
at_tbl = fp_table(df, mask_at_fp, "at_true", categorize_at_fp, Nh_map)

# ---------- SAVE ----------
out_dir = Path(OUTPUT_DIR)
out_dir.mkdir(parents=True, exist_ok=True)

def save_csv(df_in, name):
    p = out_dir / name
    df_in.to_csv(p, index=False, encoding="utf-8-sig")
    print(f"Saved -> {p}")

save_csv(yaml_tbl,  "fp_patterns_yaml_weighted.csv")
save_csv(build_tbl, "fp_patterns_build_weighted.csv")
save_csv(at_tbl,    "fp_patterns_at_weighted.csv")

# ---------- PRINT SUMMARY ----------
def pretty_top(df_in, title, k=10):
    print(f"\n=== {title} (top {k}) ===")
    if df_in is None or df_in.empty:
        print("No false positives detected for this detector.")
        return
    show = df_in[["pattern","weighted_count","share","unweighted_n","example_repos"]].head(k).copy()
    show["share"] = (show["share"]*100).round(1)
    print(show.to_string(index=False))

pretty_top(yaml_tbl,  "YAML FP patterns")
pretty_top(build_tbl, "Build FP patterns")
pretty_top(at_tbl,    "AT FP patterns")


Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\fp_patterns_yaml_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\fp_patterns_build_weighted.csv
Saved -> C:\Android Mobile App\Step3_Instr_Testing_Analysis\Type_1\Aug_10\Stratified Sample\Metrics\fp_patterns_at_weighted.csv

=== YAML FP patterns (top 10) ===
        pattern  weighted_count  share  unweighted_n                                                                                           example_repos
unknown_fp_yaml           474.0  100.0            19 zeshuaro.appainter; medic.cht-gateway; yuriykulikov.alarmclock; owntracks.android; haroldadmin.moonshot

=== Build FP patterns (top 10) ===
No false positives detected for this detector.

=== AT FP patterns (top 10) ===
                  pattern  weighted_count  share  unweighted_n                                                                                    